# Lenguajes y Autómatas II (SCD-1016)
## Sesión 07 · Evaluación de Expresiones Aritméticas y Casos Límite

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Quién evalúa las expresiones constantes: el programa o el compilador?
>
> **Objetivo de la sesión:** Evaluar expresiones de forma recursiva y detectar estáticamente la división entre cero antes de que el compilador se caiga.

**Entregable:** tu evaluador en artesano_sesion07_c.c, el harness en verde y la bitácora

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

Implementas evaluar(nodo, &error), que calcula el árbol y marca error=1 si hay división entre cero y error=2 si el nodo es inválido.

Al final de la sesión tendrás: **Un evaluador de árboles que devuelve el valor, marca el error exacto y no se cae ante una división entre cero.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 07](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion07.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion07.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion07.h"

int evaluar(ASTNode *n, int *error);
```

- Nunca ejecutes la división antes de comprobar el divisor.
- evaluar(NULL, &e) no debe caerse.
- El error se propaga hacia arriba sin recalcular los padres.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion07_c.c` junto con `artesano_sesion07_c.c`
> (o `ia_sesion07_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `(2+3)*4` | 20 |
| 2 | `7/0 -> error 1` | 1 |
| 3 | `7%0 -> error 1` | 1 |
| 4 | `arbol NULL -> error 2` | 2 |
| 5 | `10-2-3` | 5 |
| 6 | `(8/2)+1` | 5 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 07 - Evaluador recursivo con deteccion de errores ===
Caso                               esperado:              obtenido:              resultado
[ 1] (2+3)*4                            esperado: 20           obtenido: 20           OK
[ 2] 7/0 -> error 1                     esperado: 1            obtenido: 1            OK
[ 3] 7%0 -> error 1                     esperado: 1            obtenido: 1            OK
[ 4] arbol NULL -> error 2              esperado: 2            obtenido: 2            OK
[ 5] 10-2-3                             esperado: 5            obtenido: 5            OK
[ 6] (8/2)+1                            esperado: 5            obtenido: 5            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion07.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] La división y el módulo comprueban el divisor antes de operar.
- [ ] evaluar(NULL,&e) devuelve error=2 sin caerse.
- [ ] Un error en un hijo impide evaluar el padre.


## Paso 0 · Entorno


In [ ]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion07_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [ ]:
%%writefile api_sesion07.h
/* api_sesion07.h - contrato de la Sesion 07
 * Evaluador recursivo con detección de errores
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION07_H
#define API_SESION07_H

typedef enum { NODO_NUM, NODO_BINOP } TipoNodo;
typedef struct ASTNode {
    TipoNodo tipo;
    int valor;
    char op;
    struct ASTNode *izq;
    struct ASTNode *der;
} ASTNode;

int evaluar(ASTNode *n, int *error);

#endif /* API_SESION07_H */


In [ ]:
%%writefile artesano_sesion07_c.c
/* artesano_sesion07_c.c - Fase 1: El Artesano (sin IA)
 * Evaluador recursivo con detección de errores.
 *
 * Implementa lo que declara api_sesion07.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion07.h"
#include <stdio.h>
#include <string.h>

int evaluar(ASTNode *n, int *error) {
    (void)n; (void)error; /* TODO: hoja, operador e informar el error */
    return 0;
}


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion07_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [ ]:
%%writefile ia_sesion07_c.c
/* ia_sesion07_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion07.h. No puede traer main().
 */
#include "api_sesion07.h"

/* TODO: pega aqui la propuesta de la IA */


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [ ]:
%%writefile harness_sesion07_c.c
/* harness_sesion07_c.c - auditoria automatica de la Sesion 07
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion07_c.c artesano_sesion07_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion07.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 07 - Evaluador recursivo con deteccion de errores ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { ASTNode a={NODO_NUM,2,0,NULL,NULL}, b={NODO_NUM,3,0,NULL,NULL}, c={NODO_NUM,4,0,NULL,NULL}; ASTNode s={NODO_BINOP,0,'+',&a,&b}, m={NODO_BINOP,0,'*',&s,&c}; int e=0; int r=evaluar(&m,&e); (void)e; verificar_int("(2+3)*4", (long)(20), (long)(r)); }
    { ASTNode a={NODO_NUM,7,0,NULL,NULL}, b={NODO_NUM,0,0,NULL,NULL}; ASTNode d={NODO_BINOP,0,'/',&a,&b}; int e=0; int r=evaluar(&d,&e); (void)r; verificar_int("7/0 -> error 1", (long)(1), (long)(e)); }
    { ASTNode a={NODO_NUM,7,0,NULL,NULL}, b={NODO_NUM,0,0,NULL,NULL}; ASTNode d={NODO_BINOP,0,'%',&a,&b}; int e=0; int r=evaluar(&d,&e); (void)r; verificar_int("7%0 -> error 1", (long)(1), (long)(e)); }
    { int e=0; int r=evaluar(NULL,&e); (void)r; verificar_int("arbol NULL -> error 2", (long)(2), (long)(e)); }
    { ASTNode a={NODO_NUM,10,0,NULL,NULL}, b={NODO_NUM,2,0,NULL,NULL}, c={NODO_NUM,3,0,NULL,NULL}; ASTNode s={NODO_BINOP,0,'-',&a,&b}, r2={NODO_BINOP,0,'-',&s,&c}; int e=0; int r=evaluar(&r2,&e); (void)e; verificar_int("10-2-3", (long)(5), (long)(r)); }
    { ASTNode a={NODO_NUM,8,0,NULL,NULL}, b={NODO_NUM,2,0,NULL,NULL}, c={NODO_NUM,1,0,NULL,NULL}; ASTNode d={NODO_BINOP,0,'/',&a,&b}, s={NODO_BINOP,0,'+',&d,&c}; int e=0; int r=evaluar(&s,&e); (void)e; verificar_int("(8/2)+1", (long)(5), (long)(r)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion07.md" "\n");
    return 0;
}


In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion07_c.c artesano_sesion07_c.c -o test_artesano && ./test_artesano

In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion07_c.c ia_sesion07_c.c -o test_ia && ./test_ia

## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion07.md` a `auditorias/auditoria_sesion07.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
